# YOLOv8 Object Detection Model Training Pipeline

Notebook ini berisi pipeline lengkap untuk melatih model **YOLOv8 Object Detection** guna mendeteksi kesegaran bahan makanan (`fresh` / segar dan `rotten` / tidak segar). 

### Tahapan Pipeline:
1. **Import Libraries & Setup**: Memuat library yang diperlukan dan memverifikasi akselerasi perangkat keras (CPU vs GPU).
2. **Exploratory Data Analysis (EDA)**: Memeriksa statistik dataset, struktur berkas, dan memvisualisasikan sampel gambar beserta kotak labelnya.
3. **Model Initialization**: Memuat model detektor pre-trained YOLOv8 Nano (`yolov8n.pt`).
4. **Model Training**: Menjalankan proses training pada dataset dengan parameter optimasi dan regulasi anti-overfitting.
5. **Training Evaluation**: Menganalisis grafik metrik akurasi, loss, dan confusion matrix hasil pelatihan.
6. **Model Testing (Inference)**: Menguji model pada data test independen dan menampilkan visualisasi hasil deteksi kotak pembatas.

## 1. Import Libraries & Setup

Pada langkah pertama, kita menginstal library yang diperlukan (jika belum terinstal) dan mengimpor modul dasar untuk computer vision, visualisasi, dan pemodelan YOLO.

In [ ]:
# Instal dependensi utama
!pip install -r requirements.txt

In [ ]:
import os
import yaml
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

# Periksa ketersediaan GPU CUDA untuk akselerasi training
cuda_available = torch.cuda.is_available()
device = 0 if cuda_available else "cpu"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {cuda_available}")
if cuda_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
print(f"Selected YOLO training device: {device}")

## 2. Exploratory Data Analysis (EDA) & Dataset Validation

Sebelum melatih model, kita harus memvalidasi data untuk memastikan jumlah gambar seimbang, konfigurasi kelas benar, dan label koordinat bounding box dipetakan dengan tepat.

In [ ]:
# Setup Path
YOLO_VERSION_DIR = Path.cwd()
DATASET_DIR = YOLO_VERSION_DIR / "yolo_detection_dataset"
YAML_PATH = DATASET_DIR / "data.yaml"

# 1. Membaca konfigurasi data.yaml
if YAML_PATH.exists():
    with open(YAML_PATH, 'r') as f:
        dataset_info = yaml.safe_load(f)
    print("--- DATA.YAML CONFIGURATION ---")
    print(yaml.dump(dataset_info))
    classes = dataset_info.get('names', [])
else:
    print(f"Error: data.yaml tidak ditemukan di {YAML_PATH}")
    classes = ['fresh', 'rotten']

In [ ]:
# 2. Menghitung jumlah gambar pada masing-masing split dataset
splits = ['train', 'valid', 'test']
counts = {}

for split in splits:
    img_dir = DATASET_DIR / split / 'images'
    if img_dir.exists():
        img_files = list(img_dir.glob('**/*'))
        counts[split] = len([f for f in img_files if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
    else:
        counts[split] = 0

print("--- DATASET SPLIT STATS ---")
for split, count in counts.items():
    print(f"{split.capitalize()} set size: {count} images")

# Visualisasi statistik dataset
plt.figure(figsize=(8, 4))
plt.bar(counts.keys(), counts.values(), color=['#0D9488', '#F59E0B', '#3B82F6'])
plt.title('Dataset Split Distribution')
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 3. Visualisasi Sampel Gambar dengan Bounding Box Anotasi dari Roboflow
def plot_sample_with_boxes(image_path, label_path, class_names):
    # Load Image
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img.shape
    
    # Read Labels (Format YOLO: class_id x_center y_center w h, dinormalisasi 0-1)
    if label_path.exists():
        with open(label_path, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id = int(parts[0])
                x_c, y_c, w, h = map(float, parts[1:])
                
                # Kembalikan koordinat normalisasi ke piksel riil
                x1 = int((x_c - w/2) * width)
                y1 = int((y_c - h/2) * height)
                x2 = int((x_c + w/2) * width)
                y2 = int((y_c + h/2) * height)
                
                # Setel warna kotak berdasarkan kelas
                # Class 0: fresh (hijau), Class 1: rotten (merah)
                color = (13, 148, 136) if class_id == 0 else (217, 45, 32)
                label = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"
                
                # Gambar Box & Label Teks
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Visualisasi Sampel Ground-Truth (Roboflow)")
    plt.show()

# Ambil satu sampel gambar dari train set untuk ditampilkan
train_img_dir = DATASET_DIR / 'train' / 'images'
train_label_dir = DATASET_DIR / 'train' / 'labels'
sample_images = list(train_img_dir.glob('*.jpg'))

if sample_images:
    sample_img = sample_images[0]
    sample_label = train_label_dir / f"{sample_img.stem}.txt"
    plot_sample_with_boxes(sample_img, sample_label, classes)
else:
    print("Tidak ada sampel gambar di folder train untuk divisualisasikan.")

## 3. Load YOLOv8 Pre-trained Model

Kita memuat model deteksi objek **YOLOv8 Nano (`yolov8n.pt`)**. Model ini sangat ringan (~3 juta parameter), cepat untuk inferensi waktu nyata pada webcam, dan memiliki performa deteksi yang baik.

In [ ]:
# Inisialisasi model YOLOv8 Object Detection pre-trained
model_name = "yolov8n.pt"
model = YOLO(model_name)
print(f"Loaded pretrained model: {model_name}")

## 4. Execute Model Training

Proses training dijalankan dengan mengarahkan file konfigurasi `data.yaml` dan menambahkan parameter optimasi:
- `patience=5`: Jika akurasi validasi tidak meningkat dalam 5 epoch berturut-turut, training berhenti secara otomatis (Early Stopping).
- `dropout=0.15`: Mengaktifkan dropout pada lapisan kepala deteksi untuk mencegah overfitting.
- `imgsz=640`: Ukuran resolusi standar deteksi objek YOLO.

In [ ]:
RUNS_DIR = YOLO_VERSION_DIR / "runs"

print(f"Training menggunakan config dataset: {YAML_PATH}")
print(f"Training logs/weights akan disimpan di: {RUNS_DIR}")

# Memulai Pelatihan Model
results = model.train(
    data=str(YAML_PATH),
    epochs=25,             # Batas maksimal epoch pelatihan
    imgsz=640,             # Resolusi gambar deteksi objek
    batch=16,              # Ukuran batch yang ramah memori (VRAM CPU/GPU)
    workers=4,
    device=device,         # Menggunakan GPU atau CPU otomatis
    project=str(RUNS_DIR), 
    name="detect_freshness",
    patience=5,            # Hentikan pelatihan jika 5 epoch tidak membaik (Early Stopping)
    dropout=0.15,          # Regularisasi mencegah overfitting
    val=True               # Lakukan evaluasi validasi setiap selesai 1 epoch
)

## 5. Evaluate Training Results

Setelah training selesai, YOLO secara otomatis menyimpan log pelatihan dan metrik akurasi ke folder proyek. Kita akan memuat grafik evaluasi tersebut ke dalam notebook ini.

In [ ]:
from IPython.display import Image, display

runs_dir = Path.cwd() / "runs" / "detect_freshness"

print("Visualisasi Metrik Hasil Training:")
if runs_dir.exists():
    # 1. Menampilkan Hasil Kurva Precision, Recall, mAP, dan Loss
    results_plot = runs_dir / "results.png"
    if results_plot.exists():
        print("\nShowing Training Metrics & Curves:")
        display(Image(filename=str(results_plot)))
        
    # 2. Menampilkan Confusion Matrix akhir model
    cm_plot = runs_dir / "confusion_matrix.png"
    if cm_plot.exists():
        print("\nShowing Confusion Matrix:")
        display(Image(filename=str(cm_plot)))
else:
    print("Runs folder tidak ditemukan. Jalankan training terlebih dahulu!")

## 6. Model Testing & Bounding Box Inference

Mari kita lakukan uji coba inferensi langsung menggunakan model terbaik hasil latihan (`best.pt`) pada beberapa file gambar di test set untuk memverifikasi apakah model dapat mendeteksi dan mengotaki objek dengan benar sebelum diimplementasikan ke dalam aplikasi web.

In [ ]:
# Load best weights hasil training
best_weights = runs_dir / "weights" / "best.pt"

if best_weights.exists():
    # Memuat model terbaik yang telah dilatih
    best_model = YOLO(str(best_weights))
    
    # Ambil sampel dari folder test
    test_img_dir = DATASET_DIR / 'test' / 'images'
    test_images = list(test_img_dir.glob('*.jpg'))
    
    if test_images:
        # Prediksi 3 sampel acak dari test set
        sample_test_imgs = test_images[:3]
        for idx, img_path in enumerate(sample_test_imgs, 1):
            # Jalankan prediksi
            results = best_model(str(img_path), imgsz=640, verbose=False)
            
            # Simpan visualisasi hasil deteksi ke disk temporer lalu tampilkan
            for result in results:
                # Menggambar kotak pembatas hasil prediksi
                annotated_img = result.plot()
                
                # Tampilkan menggunakan matplotlib
                plt.figure(figsize=(6, 6))
                plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
                plt.title(f"Hasil Pengujian Deteksi Sampel {idx}")
                plt.axis('off')
                plt.show()
    else:
        print("Tidak ada sampel gambar di folder test untuk pengujian.")
else:
    print(f"File weights terbaik tidak ditemukan di {best_weights}. Silakan lakukan training terlebih dahulu.")